# NB3 — Induction Detection

This is the payoff notebook. We finally **find the induction heads** among all 144.

**Why induction heads matter.** They're the mechanism behind a lot of in-context learning: the
ability to notice `...A B ... A` and predict `B` — i.e. "I've seen this token before; whatever
followed it last time probably follows it again." That's how a model copies names, completes
repeated phrases, and picks up patterns from the prompt itself without any weight updates. Induction
heads were identified as a key driver of in-context learning in Anthropic's *In-context Learning and
Induction Heads* work.

**Plan:**
1. Build **repeated-random-token** sequences — the cleanest possible test for induction.
2. Measure **per-position loss** and watch it collapse on the second copy (the induction signature).
3. Understand the **induction attention offset** and see one head do it.
4. Turn that into an **induction score** over all 144 heads → a layers×heads heatmap.
5. Practice: re-derive the score with a **forward hook** — the tool we'll need for ablation in NB4.

## 0. Setup

We fix a seed because the sequences are random and we want reproducibility (a course convention).

In [ ]:
import torch
import circuitsvis as cv
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
nL, nH = model.cfg.n_layers, model.cfg.n_heads
print("model on", device, "|", nL, "layers x", nH, "heads")

## 1. Repeated-random-token sequences

Why *random* tokens? Because we want to isolate induction from ordinary language modelling. On real
text, a good next-token guess could come from grammar, world knowledge, anything. On a **random**
sequence there is *nothing* to exploit — except that we paste the **same random sequence twice**. So
any ability to predict the second copy can *only* come from copying the first copy: pure induction.

We build `[BOS, rand_seq, rand_seq]`:

```
position:  0      1 .. T         T+1 .. 2T
token:     BOS    rand[0..T-1]   rand[0..T-1]   (the SAME random tokens again)
```

Total length `1 + 2T`. The prefix is just the BOS token.

In [ ]:
def gen_repeated_tokens(model, seq_len, batch=1):
    """Return [batch, 1 + 2*seq_len] = [BOS, rand, rand] on the model's device."""
    bos = torch.full((batch, 1), model.tokenizer.bos_token_id, dtype=torch.long)
    rand = torch.randint(0, model.cfg.d_vocab, (batch, seq_len), dtype=torch.long)
    rep = torch.cat([bos, rand, rand], dim=1)          # duplicate rand along position
    return rep.to(model.cfg.device)

seq_len = 50
rep_tokens = gen_repeated_tokens(model, seq_len, batch=10)
print("rep_tokens shape:", tuple(rep_tokens.shape))    # [10, 101]

# Sanity check: the two halves are identical (ignoring the BOS at position 0).
first_copy  = rep_tokens[:, 1:seq_len + 1]
second_copy = rep_tokens[:, seq_len + 1:]
print("halves identical:", torch.equal(first_copy, second_copy))

## 2. The induction signature: per-position loss

Now run the model and look at the loss **at each position**. `return_type="loss"` with
`loss_per_token=True` gives the cross-entropy of predicting each next token, shape `[batch, seq-1]`.

What we expect:
- On the **first copy** the tokens are random and unseen → the model can't predict them → **high
  loss** (near `ln(50257) ≈ 10.8`, i.e. chance).
- On the **second copy** every token is a repeat of one it already saw → if induction works, loss
  **drops sharply**.

The cliff at the halfway point is the whole story.

In [ ]:
loss_per_token = model(rep_tokens, return_type="loss", loss_per_token=True)  # [batch, seq-1]
loss_by_pos = loss_per_token.mean(0).cpu()                                   # average over batch

first_half_mean  = loss_by_pos[:seq_len].mean().item()
second_half_mean = loss_by_pos[seq_len:].mean().item()
print(f"mean loss, first copy:  {first_half_mean:.3f}")
print(f"mean loss, second copy: {second_half_mean:.3f}")

plt.figure(figsize=(8, 3))
plt.plot(loss_by_pos)
plt.axvline(seq_len - 0.5, color="red", ls="--", label="second copy begins")
plt.xlabel("position"); plt.ylabel("loss"); plt.legend(); plt.title("Per-position loss on [BOS, rand, rand]")
plt.show()

## 3. The induction attention offset

*How* does the model pull this off? An **induction head**, sitting at a destination position `d` in
the second copy (current token `X`), looks back to the **previous occurrence** of `X` in the first
copy and attends to the token **right after** it — because that's what should come next.

Let's put that offset in coordinates. If `X` sits at position `d` in the second copy, its first
occurrence is at `d - T` (one full period earlier), and the token after it is at `d - T + 1`. So the
induction head attends **from `d` to `d - T + 1`** — a diagonal stripe at offset

```
src - dest = (d - T + 1) - d = -(T - 1) = -(seq_len - 1)
```

So: **current-token = offset 0, previous-token = offset −1, induction = offset −(seq_len−1)**. Same
diagonal trick from NB2, just a bigger offset. Let's *see* it on **L5H5** (our top-scoring head):
in the widget, pick head 5 and notice the bright stripe far below the diagonal, in the first-copy
region.

In [ ]:
# Cache one sequence so we can look at attention patterns.
rep_one = gen_repeated_tokens(model, seq_len, batch=1)
_, cache = model.run_with_cache(rep_one)
str_tokens = model.to_str_tokens(rep_one[0])

# L5H5's induction stripe as a single number (mean over the offset diagonal):
p = cache["pattern", 5][0]                                    # [head, dest, src]
stripe = p[5].diagonal(offset=-(seq_len - 1))                 # head 5's induction diagonal
print(f"L5H5 induction-stripe mean: {stripe.mean().item():.3f}")

# Visualize all of layer 5; select head 5 to see the stripe.
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", 5][0])

## 4. Induction score over all 144 heads

Now scale it up: compute the induction-stripe mean for **every** head and plot a layers×heads
heatmap. This is the same cache + diagonal approach you already know from NB2, just with
`offset=-(seq_len-1)` and averaged over the batch too.

A head with a high score is one that consistently attends to "the token after the previous copy" —
our operational definition of an induction head.

In [ ]:
# Cache the full batch so the score averages over 10 sequences (less noisy).
_, cache_batch = model.run_with_cache(rep_tokens)

induction_score = torch.zeros(nL, nH)
for L in range(nL):
    p = cache_batch["pattern", L]                            # [batch, head, dest, src]
    stripe = p.diagonal(offset=-(seq_len - 1), dim1=-2, dim2=-1)   # [batch, head, diag]
    induction_score[L] = stripe.mean(dim=(0, -1)).cpu()      # mean over batch AND diagonal

plt.figure(figsize=(8, 5))
plt.imshow(induction_score.numpy(), aspect="auto", cmap="viridis", origin="upper")
plt.colorbar(label="induction score"); plt.xlabel("head"); plt.ylabel("layer")
plt.title("Induction score per head (GPT-2 small)")
plt.xticks(range(nH)); plt.yticks(range(nL)); plt.show()

flat = induction_score.flatten()
print("Top induction heads:")
for i in flat.topk(5).indices:
    print(f"  L{int(i)//nH}H{int(i)%nH}: {flat[i]:.3f}")

## 5. Practice — do it with a forward hook

So far we've computed everything by first caching *all* activations and indexing afterward. But
TransformerLens's real power is **hooks**: functions that run *during* the forward pass and can read
(or later, edit) an activation in place. You'll need this for ablation in NB4, so let's practice the
read-only version here by re-deriving the induction score.

**The hook function signature** is always `(activation, hook)`:
- `activation` — the tensor flowing through that hook point (here `pattern`, `[batch, head, dest, src]`).
- `hook` — metadata about the hook point; `hook.layer()` gives the layer index.

You register hooks with `model.run_with_hooks(tokens, return_type=None, fwd_hooks=[(where, fn), ...])`,
where `where` is a name (or a filter function that returns `True` for the hook names you want).

**Your task:** fill in `induction_score_hook` so it writes each layer's per-head induction score into
`store`. Then run it and confirm you get the **same top heads** (L5H5, L7H10, L6H9, L5H1, ...) as the
cache-based version above.

Hints:
- The stripe is `activation.diagonal(offset=-(seq_len-1), dim1=-2, dim2=-1)` → `[batch, head, diag]`.
- Reduce to `[head]` with `.mean(dim=(0, -1))`.
- Write it into `store[hook.layer()]`.
- The name filter for attention patterns is `lambda name: name.endswith("hook_pattern")`.

In [ ]:
store = torch.zeros(nL, nH, device=device)

def induction_score_hook(pattern, hook):
    # pattern: [batch, head, dest, src]
    # TODO(you):
    # stripe = ...                       # diagonal at offset -(seq_len-1)
    # score  = ...                       # mean over batch and diagonal -> [head]
    # store[hook.layer()] = score
    pass

# TODO(you): run the model with the hook attached to every attention-pattern hook point.
# model.run_with_hooks(
#     rep_tokens,
#     return_type=None,
#     fwd_hooks=[(lambda name: name.endswith("hook_pattern"), induction_score_hook)],
# )

# flat = store.flatten()
# for i in flat.topk(5).indices:
#     print(f"L{int(i)//nH}H{int(i)%nH}: {flat[i]:.3f}")

<details>
<summary>Reference solution (open after you've tried)</summary>

```python
store = torch.zeros(nL, nH, device=device)

def induction_score_hook(pattern, hook):
    # pattern: [batch, head, dest, src]
    stripe = pattern.diagonal(offset=-(seq_len - 1), dim1=-2, dim2=-1)  # [batch, head, diag]
    store[hook.layer()] = stripe.mean(dim=(0, -1))                      # [head]

model.run_with_hooks(
    rep_tokens,
    return_type=None,
    fwd_hooks=[(lambda name: name.endswith("hook_pattern"), induction_score_hook)],
)

flat = store.flatten()
for i in flat.topk(5).indices:
    print(f"L{int(i)//nH}H{int(i)%nH}: {flat[i]:.3f}")
```

This should match the cache-based ranking: L5H5, L7H10, L6H9, L5H1, L7H2.

</details>

---
**Done?** When your hook reproduces the top induction heads, you've built the detector the whole
course was aiming at. Ask me to review.

Next, **NB4 — circuit decomposition**: we'll connect the **previous-token head (L4H11)** from NB2 to
these **induction heads** via K-composition, then **ablate** them (zero / mean) and watch the second
copy's loss shoot back up — proving these heads are *necessary*, not just correlated. We'll finish
with **direct logit attribution** to measure each head's contribution to the correct next token.